# Projectile Predictor model training (Minecraft Java 26.2)

This notebook creates a synthetic training set for a fast angle estimator. Inputs include displacement, target velocity, the shooter's instantaneous X/Y/Z velocity, and projectile physics. Labels come from a numerical ballistic solver that assumes the projectile inherits the full player velocity vector. This is a baseline only; validate against measured game trajectories before claiming accuracy.

Capture rows later with projectile, dx, dy, dz, target_vx, target_vy, target_vz, shooter_vx, shooter_vy, shooter_vz, speed, gravity, drag, yaw, and pitch.

In [1]:
import json, math, os, random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED=2602; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
OUT='/kaggle/working'; os.makedirs(OUT, exist_ok=True)
print('TensorFlow', tf.__version__)

2026-10-03 17:50:14.517522: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791049814.863382      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791049814.964702      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791049815.809067      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791049815.809114      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791049815.809117      16 computation_placer.cc:177] computation placer alr

TensorFlow 2.19.0


## Generate synthetic labels
This iterative low-arc solver assumes constant gravity and ignores drag. The generated rows demonstrate the Kaggle-to-JSON pipeline; they are not measurements of Minecraft physics.

In [2]:
def solve(dx,dy,dz,vx,vy,vz,svx,svy,svz,speed,g):
    # Assume full inheritance of player instantaneous motion on X, Y, and Z.
    vx,vy,vz=vx-svx,vy-svy,vz-svz
    t=math.sqrt(dx*dx+dz*dz)/max(speed,1e-6)
    for _ in range(12):
        x,y,z=dx+vx*t,dy+vy*t,dz+vz*t; r=math.hypot(x,z)
        if r<1e-5: return None
        v2=speed*speed; disc=v2*v2-g*(g*r*r+2*y*v2)
        if disc<=0: return None
        tanp=(v2-math.sqrt(disc))/(g*r) if g>1e-9 else y/r
        pitch=math.atan(tanp); nt=r/max(speed*math.cos(pitch),1e-6)
        if not math.isfinite(nt) or nt>8: return None
        if abs(nt-t)<1e-4: t=nt; break
        t=.5*t+.5*nt
    yaw=math.atan2(-dx-vx*t,dz+vz*t)
    return yaw,pitch,t

profiles={'arrow':(3,.05,.99),'trident':(2.5,.05,.99),'snowball':(1.5,.03,.99),'egg':(1.5,.03,.99),'ender_pearl':(1.5,.03,.99),'potion':(.5,.05,.99),'wind_charge':(1.5,0,1),'custom_fast':(4,.02,.985)}
rows=[]
for kind,(base_speed,base_g,base_drag) in profiles.items():
    for _ in range(9000):
        dx,dz=random.uniform(-60,60),random.uniform(6,90); dy=random.uniform(-12,18)
        vx,vy,vz=random.uniform(-.22,.22),random.uniform(-.08,.08),random.uniform(-.22,.22)
        svx,svy,svz=random.uniform(-.35,.35),random.uniform(-.18,.18),random.uniform(-.35,.35)
        speed=base_speed*random.uniform(.8,1.15); g=base_g*random.uniform(.85,1.15); drag=base_drag*random.uniform(.985,1.005)
        result=solve(dx,dy,dz,vx,vy,vz,svx,svy,svz,speed,g)
        if result is None: continue
        yaw,pitch,flight_time=result
        rows.append({'projectile':kind,'dx':dx,'dy':dy,'dz':dz,'target_vx':vx,'target_vy':vy,'target_vz':vz,'shooter_vx':svx,'shooter_vy':svy,'shooter_vz':svz,'speed':speed,'gravity':g,'drag':drag,'yaw':yaw,'pitch':pitch,'flight_time':flight_time})
df=pd.DataFrame(rows); df.to_csv(f'{OUT}/projectile_dataset.csv',index=False)
print(df.shape, df.projectile.value_counts().to_dict()); df.head()

(1673, 16) {'custom_fast': 919, 'arrow': 380, 'trident': 236, 'egg': 38, 'wind_charge': 36, 'snowball': 32, 'ender_pearl': 32}


,projectile,dx,dy,dz,target_vx,target_vy,target_vz,shooter_vx,shooter_vy,shooter_vz,speed,gravity,drag,yaw,pitch,flight_time
0,arrow,2.216658,-4.904710,23.857411,-0.127667,-0.048275,-0.007262,-0.296438,-0.097566,0.301688,3.020877,0.050388,0.987707,-0.158476,-0.145071,7.316842
1,arrow,7.981471,8.413715,9.116607,0.163246,0.036046,-0.067295,-0.218711,-0.016489,0.193979,3.390638,0.046748,0.979867,-0.887578,0.629823,4.578422
2,arrow,17.225209,-2.972175,10.119909,-0.149155,0.063380,-0.053407,0.015247,-0.175176,0.152600,3.401104,0.048496,0.985891,-1.067392,-0.050061,5.485992
3,arrow,-5.375072,0.908205,11.373246,0.012244,0.045293,0.075321,0.317539,-0.023331,0.116478,2.463435,0.042955,0.979568,0.562211,0.143681,5.405561
4,arrow,-7.222549,-10.806946,6.764400,-0.110030,-0.002901,0.164167,0.054149,-0.002833,0.264679,3.144302,0.044826,0.975513,0.902382,-0.793385,4.609477


## Train and evaluate
Report angle MAE in degrees. For serious evaluation, split by entire projectile classes, distances, and captured sessions rather than random rows.

In [3]:
features=['dx','dy','dz','target_vx','target_vy','target_vz','shooter_vx','shooter_vy','shooter_vz','speed','gravity','drag']; targets=['yaw_sin','yaw_cos','pitch']
df['yaw_sin']=np.sin(df['yaw']); df['yaw_cos']=np.cos(df['yaw'])
X=df[features].to_numpy(np.float32); y=df[targets].to_numpy(np.float32)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.15,random_state=SEED)
scaler=StandardScaler().fit(Xtr); Xtr=scaler.transform(Xtr).astype(np.float32); Xte=scaler.transform(Xte).astype(np.float32)
model=tf.keras.Sequential([tf.keras.layers.Input((len(features),)),tf.keras.layers.Dense(64,activation='relu'),tf.keras.layers.Dense(64,activation='relu'),tf.keras.layers.Dense(3)])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mse')
model.fit(Xtr,ytr,validation_split=.1,epochs=30,batch_size=512,callbacks=[tf.keras.callbacks.EarlyStopping(patience=4,restore_best_weights=True)],verbose=1)
pred=model.predict(Xte,verbose=0); predicted_yaw=np.arctan2(pred[:,0],pred[:,1]); actual_yaw=np.arctan2(yte[:,0],yte[:,1]); dyaw=np.arctan2(np.sin(predicted_yaw-actual_yaw),np.cos(predicted_yaw-actual_yaw))
metrics={'yaw_mae_degrees':float(np.degrees(np.mean(np.abs(dyaw)))),'pitch_mae_degrees':float(np.degrees(np.mean(np.abs(pred[:,2]-yte[:,2])))),'rows':int(len(df))}
print(json.dumps(metrics,indent=2))

Epoch 1/30


2026-10-03 17:50:49.700261: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - loss: 0.4622 - val_loss: 0.3338
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.3247 - val_loss: 0.2426
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.2380 - val_loss: 0.1911
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1893 - val_loss: 0.1624
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.1622 - val_loss: 0.1417
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1421 - val_loss: 0.1218
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1225 - val_loss: 0.1027
Epoch 8/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.1041 - val_loss: 0.0873
Epoch 9/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.0893 - val_loss: 0.0766
Epoch 10/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0790 - val_loss: 0.0699
Epoch 11/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.0725 - val_loss: 0.0656
Epoch 12/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0682 - val_loss: 0.0619
Epoch 13/30

## Export model weights for Java inference
Evaluate each dense layer as Wx+b; apply ReLU where marked. The mod still needs an item-to-physics mapping. A model cannot infer arbitrary modded item behavior from an item name.

In [4]:
layers=[]
for layer in model.layers:
    w,b=layer.get_weights(); layers.append({'weights':w.astype(float).tolist(),'bias':b.astype(float).tolist(),'activation':getattr(layer.activation,'__name__','linear')})
artifact={'format':'projectile-predictor-mlp-v1','features':features,'targets':targets,'mean':scaler.mean_.astype(float).tolist(),'scale':scaler.scale_.astype(float).tolist(),'layers':layers,'metrics':metrics,'training_data':'synthetic-vacuum-ballistics-v1','warning':'Synthetic labels are not an in-game accuracy guarantee; calibrate with captures.'}
with open(f'{OUT}/projectile_model.json','w') as f: json.dump(artifact,f,separators=(',',':'))
print('Wrote',f'{OUT}/projectile_model.json')

Wrote /kaggle/working/projectile_model.json


## Replace synthetic data with captured game trajectories
Capture actual item launch parameters and resulting projectile positions every tick in a controlled world. Include bow charge, crossbow charged state, enchantments, potion type, shooter motion, and target state. Split test data by session and projectile type. Prefer exact vanilla launch behavior where available; train only residual corrections for effects the solver misses.